# F2 · When the forest loses: linear relationship, trend and cost

**Type**: Compare · **Level**: ●●● challenge · **Time**: about 60 minutes · **extended path**

## Objective

Identify the three situations in which a random forest is not the best choice, comparing it with linear/logistic regression: (1) a **linear** relationship with many features (the linear model wins and costs a hundredth), (2) **extrapolation** of a trend (the forest is flat), (3) **cost** of training, prediction and memory
(the forest is hundreds of times heavier). And the situation in which it wins: nonlinear interactions.

## Dataset

Three problems (`problems()`, given): `linear` (3000 examples, 60 features of which 15 linear), `xor` (2000 examples, `y = (x0 > 0) XOR (x1 > 0)` with 8 noise features), `trend` (series with linear trend and seasonality, training on the first 300 months, test on the next 100). Models: logistic (or linear), random forest of 200 trees.

## Educational purpose

Develop the criterion for choosing: the forest is an excellent default but not a universal one. Measure for each problem accuracy (or RMSE), `fit` time, `predict` time and model size.

## Task

1. For `linear` and `xor` (classification, 60/40 split with `random_state=0`) train `make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))` and `RandomForestClassifier(200, random_state=0, n_jobs=-1)`: save `cls_table` (index `(problem, model)`, columns `acc`, `t_fit`, `t_predict`, `mb`) with `mb = len(pickle.dumps(model)) / 1e6`.
2. For `trend` (regression) train `LinearRegression` and `RandomForestRegressor(200, random_state=0, n_jobs=-1)` on the first 300 months with features `t`, `sin`, `cos`; save `trend_rmse` (dictionary model → RMSE on the next 100 months).
3. Save `linear_wins_on` (sorted list of the problems where the linear/logistic model is better or on par, that is within 0.01 of accuracy or with a lower RMSE), `forest_wins_on` (the problems where the forest beats the logistic by over 0.1), `fit_ratio` (forest `t_fit` / logistic `t_fit` on `linear`) and `mb_ratio`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import pickle
import time
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def problems():
    rng = np.random.default_rng(0)
    Xl = rng.normal(size=(3000, 60))
    w = np.r_[rng.normal(size=15), np.zeros(45)]
    yl = rng.binomial(1, 1 / (1 + np.exp(-2.5 * Xl @ w / np.linalg.norm(w))))
    Xx = rng.normal(size=(2000, 10))
    yx = ((Xx[:, 0] > 0) ^ (Xx[:, 1] > 0)).astype(int)
    yx = np.where(rng.random(2000) < 0.05, 1 - yx, yx)
    t = np.arange(400)
    y_t = 5 + 0.05 * t + 2 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 0.5, 400)
    F = np.c_[t, np.sin(2 * np.pi * t / 12), np.cos(2 * np.pi * t / 12)]
    return {"linear": (Xl, yl), "xor": (Xx, yx)}, (F, y_t)


classification, (F_trend, y_trend) = problems()

## Your work

In [ ]:
rows = {}
for name, (X, y) in classification.items():
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    models = {"logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), "forest": RandomForestClassifier(200, random_state=0, n_jobs=-1)}
    for model_name, m in models.items():
        ...
cls_table = pd.DataFrame(rows).T
cls_table.round(3)

In [ ]:
trend_rmse = {}

In [ ]:
linear_wins_on = None
forest_wins_on = None
fit_ratio = None
mb_ratio = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`t0 = time.time(); m.fit(X_tr, y_tr); t_fit = time.time() - t0`; `t0 = time.time(); acc = m.score(X_te, y_te); t_predict = time.time() - t0`; `rows[(name, model_name)] = {...}`. Trend: `tr = F_trend[:, 0] < 300`; `LinearRegression().fit(F_trend[tr], y_trend[tr])`; RMSE on `~tr`.'):
    check(cls_table.shape == (4, 4) and list(cls_table.columns) == ["acc", "t_fit", "t_predict", "mb"], "4 rows (2 problems x 2 models) and columns acc, t_fit, t_predict, mb", "one row per (problem, model) pair")
    check(set(trend_rmse) == {"linear", "forest"} and trend_rmse["linear"] < 1 and trend_rmse["forest"] > 2.5, "trend: the linear one is off by less than 1, the forest by more than 2.5", "test on months 300-399")
    check(linear_wins_on == ["linear", "trend"] and forest_wins_on == ["xor"], "the linear model wins (or ties) on 'linear' and 'trend'; the forest beats the logistic by over 0.1 on 'xor'", "compare the accuracies and the RMSEs")
    check(cls_table.loc[("xor", "logistic"), "acc"] < 0.6 and cls_table.loc[("xor", "forest"), "acc"] > 0.9, "on the XOR the logistic is at chance (below 0.6), the forest over 0.9", "the XOR is not linear")
    check(fit_ratio > 20 and mb_ratio > 100, "the forest costs over 20 times in training and over 100 times in memory compared with the logistic", "t_fit and mb")
done()

---
The commented solution is in `solutions/F_compare/F2_when_the_forest_loses.ipynb`. Open it after you pass the check, or after you have really tried.